# Phase 6 — Qwen 2.5 7B Extraction on Colab

Self-contained Colab notebook that serves Qwen 2.5 7B Instruct via vLLM and runs the
Agentic-Healthcare-Maps extraction pipeline against the 30-record gold validation set
and (optionally) the full 10K dataset.

**Prerequisites**
- Colab Pro account with a T4 / A100 / L4 GPU runtime
- (Optional) ngrok account if you want to call the endpoint from your laptop instead of from inside Colab
- Repo URL: replace `REPO_URL` in the clone cell with your fork's HTTPS clone URL
  (you can also `Files → Upload` a zipped copy if the repo is private)

**Step list**
1. Install vLLM and runtime deps (~3–4 min)
2. Start the vLLM OpenAI-compatible server, wait for readiness (~3–5 min including model download)
3. (Optional) ngrok tunnel for off-Colab access
4. Smoke-test the endpoint
5. **STOP HERE** — manual validation gate
6. Clone the repo and run the 30-record Qwen validation pass via `agent.extract_open`
7. **GATE** — only proceed if recall ≥ 0.75 against the gold set
8. Run the full 10K extraction (overnight on T4, a couple of hours on A100)
9. Download `data/phase6_extractions_qwen.jsonl` back to your laptop

**Cost cap: $0.** This notebook does not call paid APIs.

The Sonnet 4.6 30-record reference set in `data/phase2_extractions.jsonl` stays untouched and is the comparison artifact for the demo.

In [ ]:
# Step 1 — install vLLM + the project's extraction-time deps.
# Skip pip output noise; takes ~3-4 min on a fresh runtime.
!pip install -q vllm pyngrok openai instructor pydantic rapidfuzz pandas tqdm python-dotenv mlflow opentelemetry-api opentelemetry-sdk PyYAML

In [ ]:
# Step 2 — start vLLM in the background, serving AWQ 4-bit quantized Qwen 2.5 7B.
#
# Why AWQ: T4 GPU is 14.56 GiB usable. fp16 Qwen 7B weighs 14.25 GiB on its own
# and there's no room left for KV cache or torch.compile autotuning workspace
# (~1 GiB), so the engine OOMs during the profile_run step. AWQ 4-bit drops
# the model to ~5.2 GiB and leaves ~7.3 GiB for KV cache + headroom.
# Recall on structured extraction is within ~1 pt of fp16 in published evals.
import subprocess, sys, time, os, requests

MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct-AWQ'
VLLM_PORT = 8000
VLLM_LOG = '/content/vllm_server.log'

vllm_cmd = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL_NAME,
    '--port', str(VLLM_PORT),
    '--host', '0.0.0.0',
    '--max-model-len', '4096',
    '--gpu-memory-utilization', '0.90',  # safe higher with quantized weights
    '--quantization', 'awq',
    '--dtype', 'half',
    '--enforce-eager',                    # skip torch.compile autotuning (avoids 1 GiB scratch alloc)
    '--disable-log-stats',
]
log_handle = open(VLLM_LOG, 'w')
vllm_proc = subprocess.Popen(vllm_cmd, stdout=log_handle, stderr=subprocess.STDOUT)
print(f'vLLM started, pid={vllm_proc.pid}. Waiting for /v1/models...')

READY = False
for attempt in range(600):  # up to 10 min — first run includes model download
    try:
        r = requests.get(f'http://localhost:{VLLM_PORT}/v1/models', timeout=2)
        if r.status_code == 200:
            print(f'\nReady after ~{attempt}s.')
            print(r.json())
            READY = True
            break
    except Exception:
        pass
    if attempt and attempt % 30 == 0:
        print(f'  ...{attempt}s — last 6 log lines:')
        try:
            for line in open(VLLM_LOG).readlines()[-6:]:
                print('    ' + line.rstrip())
        except Exception:
            pass
    time.sleep(1)

if not READY:
    raise RuntimeError(
        f'vLLM did not become ready in 10 min. tail of {VLLM_LOG}:\n'
        + open(VLLM_LOG).read()[-4000:]
    )

VLLM_ENDPOINT_URL = f'http://localhost:{VLLM_PORT}/v1'
print(f'\nLocal endpoint URL: {VLLM_ENDPOINT_URL}')

In [ ]:
# Step 3 (optional) — ngrok tunnel for off-Colab access.
# Skip this entire cell if you plan to run the extraction inside this notebook.
# To enable: paste your ngrok auth token below (get one free at https://dashboard.ngrok.com/get-started/your-authtoken).

NGROK_AUTH_TOKEN = ''  # <-- paste here, or leave empty to skip

if NGROK_AUTH_TOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    tunnel = ngrok.connect(VLLM_PORT, 'http')
    PUBLIC_VLLM_URL = f'{tunnel.public_url}/v1'
    print(f'ngrok tunnel up. Public endpoint: {PUBLIC_VLLM_URL}')
    print('Set this in your laptop\'s .env as VLLM_ENDPOINT_URL=' + PUBLIC_VLLM_URL)
else:
    PUBLIC_VLLM_URL = None
    print('No ngrok token provided. Will run extraction inside this notebook against localhost.')

In [ ]:
# Step 4 — smoke test. Direct OpenAI-compatible call to verify the endpoint speaks JSON.
import openai

smoke_client = openai.OpenAI(base_url=VLLM_ENDPOINT_URL, api_key='EMPTY')
smoke_resp = smoke_client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {'role': 'system', 'content': 'Respond ONLY with a JSON object of the form {"answer": <string>}.'},
        {'role': 'user', 'content': 'What is the capital of India? Reply in JSON.'},
    ],
    max_tokens=80,
    response_format={'type': 'json_object'},
)
print('smoke response:', smoke_resp.choices[0].message.content)
print('usage:', smoke_resp.usage)

## STOP HERE — validate before proceeding

Confirm before running the next cell:
1. The smoke-test cell above returned a valid JSON object (not a parse error or stack trace).
2. `usage` shows non-zero `prompt_tokens` and `completion_tokens`.
3. (If you set up ngrok) the `Public endpoint` URL printed above is reachable from your laptop with `curl <PUBLIC_VLLM_URL>/models`.

Only then run the validation pass below. Halt here if anything looks off — restarting vLLM is faster than chasing bad output.

In [ ]:
# Step 6a — clone the repo (replace REPO_URL with your fork) OR upload manually via Files panel.
import os, subprocess, shutil

REPO_URL = 'https://github.com/<your-username>/Agentic-Healthcare-Maps.git'  # <-- edit me
REPO_DIR = '/content/Agentic-Healthcare-Maps'

if not os.path.exists(REPO_DIR):
    if '<your-username>' in REPO_URL:
        raise RuntimeError(
            'REPO_URL placeholder not edited. Either set REPO_URL above, OR upload a zipped repo via '
            'the Files panel and unzip into /content/Agentic-Healthcare-Maps.'
        )
    subprocess.check_call(['git', 'clone', REPO_URL, REPO_DIR])

os.chdir(REPO_DIR)
print('cwd:', os.getcwd())
print(os.listdir('.'))

In [ ]:
# Step 6b — run the 30-record validation pass against gold_labels.jsonl.
# Wall clock estimate: ~5-15 minutes on T4 (30 records × N=5 samples × ~2-5s each).
import os
os.environ['VLLM_ENDPOINT_URL'] = VLLM_ENDPOINT_URL
os.environ['QWEN_MODEL'] = MODEL_NAME

!python -m agent.extract_open \
    --records-from data/gold_labels.jsonl \
    --output data/phase6a_qwen_validation.jsonl

In [ ]:
# Step 7 — compare Qwen output to Sonnet baseline + gold. Halts non-zero if recall < 0.75.
import subprocess
result = subprocess.run([sys.executable, 'eval/phase6a_qwen_validation.py'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr, file=sys.stderr)
if result.returncode != 0:
    raise RuntimeError(
        f'phase6a_qwen_validation.py exited with {result.returncode}. '
        f'Master prompt directive: do NOT proceed to full 10K extraction. '
        f'Inspect the output above and discuss fallback options.'
    )

## GATE — Qwen recall ≥ 0.75 confirmed?

If the cell above raised `RuntimeError`, **do not run the next cell**. Halt the notebook here and paste the validation table back to chat to discuss fallback options:
- Bump to Qwen 2.5 14B or 32B on a higher-tier GPU
- Switch to Gemini Flash free tier (1500 requests/day)
- Accept a stratified subset on Sonnet within budget

Otherwise, proceed.

In [ ]:
# Step 8 — full 10K extraction. Resumable: if interrupted, re-run this cell to continue.
# Wall-clock estimate: ~3 hr on A100, 8-15 hr on T4.
# Output: data/phase6_extractions_qwen.jsonl

!python -m agent.extract_open \
    --output data/phase6_extractions_qwen.jsonl

In [ ]:
# Step 9 — pull outputs back to your laptop.
from google.colab import files
import os, glob

for path in [
    'data/phase6a_qwen_validation.jsonl',
    'data/phase6_extractions_qwen.jsonl',
]:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1024 / 1024
        print(f'downloading {path}  ({size_mb:.1f} MB)')
        files.download(path)
    else:
        print(f'(not present: {path})')